# Visit mapping sets: demo

Concept: working note `claude_notes/visit-mapping-sets.md` (sections 3 to 7, outside this repo), based on `overview/Single Level Visit Mappings.md`.
Code: package `ocelescope_module_location_based_ocpm.visit_mapping`; this notebook only imports and calls its functions.
Further detail: the docstrings (e.g. `help(sweep_trace)`) and `docs/resources/visit-mapping.md`.

The maximal visits $\mathcal{V}_{max}$ (`visits.ipynb`) may overlap or contain each other.
A **mapping set** $\mathcal{V}_{map}$ is disjoint and covering: every relation lies in exactly one visit.
Per handling unit this is a segmentation of its trace into consecutive visits (a visit sequence).
A **strategy** decides which segmentation is taken.

What the notebook does:
1. loads the extended running example (sheets b1 to b8, slabs s1, s2) and the extra traces x1 to x5, y1 to y5,
2. derives the maximal visits and splits them into one trace per handling unit,
3. shows the sweep with the local strategies S1 to S4,
4. shows the global strategies S5 and S6 and how to define your own (criteria in an order, or weighted),
5. runs everything in one call and compares the strategies with Nina's targets,
6. evaluates mapping sets with the metrics, for all, several or one handling unit,
7. draws $\mathcal{V}_{max}$ and $\mathcal{V}_{map}$ in the trace figure, with the metrics,
8. runs S6 on the extra traces,
9. saves the S6 mapping set.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/visit_mapping.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$; durations are polars durations (e.g. `10h`).
All functions take and return polars frames; they work on eager (`DataFrame`) and lazy (`LazyFrame`) frames alike.

In [ ]:
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import MAPPING_SCHEMA
from ocelescope_module_location_based_ocpm.visit_mapping.mapping_files import load_mapping, save_mapping
from ocelescope_module_location_based_ocpm.visit_mapping.metrics import (
    mapping_metrics,
    metric_lines,
    trace_metric_lines,
)
from ocelescope_module_location_based_ocpm.visit_mapping.optimal import (
    CRITERIA,
    lexicographic,
    optimal_segmentation,
    segment_cost,
    segment_stats,
    valid_ends,
    weighted,
)
from ocelescope_module_location_based_ocpm.visit_mapping.pipeline import (
    compare_strategies,
    derive_mapping_set,
    map_traces,
)
from ocelescope_module_location_based_ocpm.visit_mapping.segments import (
    mapping_visits,
    segments_frame,
    visit_sequences,
)
from ocelescope_module_location_based_ocpm.visit_mapping.strategies import (
    STRATEGIES,
    cost_optimal,
    nina_strategy,
)
from ocelescope_module_location_based_ocpm.visit_mapping.sweep import (
    SweepStep,
    absorb_never,
    earlier_wins,
    excursions,
    later_wins,
    sweep_trace,
    vehicle_only,
)
from ocelescope_module_location_based_ocpm.visit_mapping.traces import (
    add_mapping_inputs,
    split_traces,
    storage_high_levels,
)
from ocelescope_module_location_based_ocpm.visits.context import location_context
from ocelescope_module_location_based_ocpm.visits.maximal import derive_maximal_visits
from ocelescope_module_location_based_ocpm.visits.trace_figure import relations_for_figure, trace_figure

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "mapping_example"
DERIVED = DATA / "derived"

# Show whole frames.
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_fmt_table_cell_list_len(8)
pl.Config.set_fmt_str_lengths(40);

## 1. Inputs

`data/mapping_example/` (generated by `make_mapping_example.py`, the single source of its files) holds:
- `relations.csv`: the running example extended by sheets (Level 3; 10 handling units, 41 events, 54 relations), in the format of a saved level mapping (`loc` given directly);
- `extra_relations.csv`: the extra traces x1 to x5 and y1 to y5 of the working note (6.5, 6.7);
- `locations.csv`: A1, A2 → A, B1 → B, the truck T is not mapped; $V = \{T\}$, $W = \{B1\}$;
- `processing_events.csv`: the processing operations $P$ (here e1, e3, e4, e15, e22, e26, e35 and the starred events of the extra traces).

Event eK happens at hour K. The tolerance is $\kappa_{max} = 3$, as in the working note.

In [ ]:
KAPPA = 3
relations = pl.read_csv(DATA / "relations.csv", schema=MAPPING_SCHEMA)
extra = pl.read_csv(DATA / "extra_relations.csv", schema=MAPPING_SCHEMA)
context = location_context(pl.read_csv(DATA / "locations.csv"))
P = pl.read_csv(DATA / "processing_events.csv")["event"].to_list()
print(context)
print("P =", P)
relations.select("hu", "event", "time", "activity", "loc")

## 2. Maximal visits and traces

### Maximal visits: `derive_maximal_visits(relations, location, agg, kappa, vehicles, ignore_vehicles, processing)`

As in `visits.ipynb`. The relations get their trace position (AS4 order), high-level location and `visit_id`, the id of the maximal visit $m(p)$ at the relation's own high-level location.

### Inputs of the strategies: `add_mapping_inputs(relations, vehicles=(), processing=None, location="loc")`

Adds `on_vehicle` ($loc \in V$) and `is_processing_event` ($e \in P$).

| parameter | meaning | default |
|---|---|---|
| `relations` | relations of `derive_maximal_visits` | |
| `vehicles` | the set $V$ | none |
| `processing` | $P$ as event ids | the transformation events $T$ |
| `location` | column of loc | `"loc"` |

In [ ]:
rel, maximal = derive_maximal_visits(relations, "loc", context.agg, KAPPA, processing=P)
rel = add_mapping_inputs(rel, context.vehicles, P)
maximal.select("visit_id", "hu", "high_level", "first_event", "last_event", "n_relations", "interrupted")

### One trace per handling unit: `split_traces(relations, visits, location="loc", handling_units=None, storage_high_levels=())`

The strategies work on one trace at a time. A `Trace` holds the lists of the trace (events, times, locations, high-level locations, `on_vehicle`, `is_processing`) indexed by the position, starting at 0, and for every position its maximal visit $m(p)$ (a `MaximalVisit` with 0-based first and last position).
`storage_high_levels` are the high-level locations containing a storage unit, $\{agg(w) \mid w \in W\}$ (`storage_high_levels(storage_units, agg)`; here B), used by the storage tie-break of S6.
Only handling units with an interrupted maximal visit can change; all others keep $\mathcal{V}_{max}(hu)$ under every strategy.

b7 = e26, e31, e32, e33, e34 at A2, B1, T, A2, B1: its maximal visits at A (e26–e33) and B (e31–e34) overlap.

In [ ]:
traces = {t.hu: t for t in split_traces(rel, maximal, storage_high_levels=storage_high_levels(context.storage_units, context.agg))}
b7 = traces["b7"]
pl.DataFrame({
    "position": range(len(b7)),
    "event": b7.events,
    "loc": b7.locations,
    "high_level": b7.high_levels,
    "on_vehicle": b7.on_vehicle,
    "is_processing": b7.is_processing,
    "m(p)": [f"{m.high_level} {m.first}–{m.last}" for m in b7.maximal],
})

## 3. Local strategies: the sweep

### Excursions: `excursions(trace, visit, after)`

The maximal runs of relations not at the visit's high-level location, strictly inside it, that start after position `after` (0-based first and last position).
For b7's visit to A (positions 0–3), the excursion ⟨e31, e32⟩ at positions 1–2:

In [ ]:
excursions(b7, b7.maximal[0], after=0)

### The sweep: `sweep_trace(trace, rule)`

Start at position $p = 0$; take $m(p)$ and walk along it from $p$. At each excursion $\xi$ of $m(p)$ after $p$ the rule decides: **absorb** (continue) or **split** (the visit ends before $\xi$). Output the segment from $p$ to where the walk stopped, and continue after it.
The result always partitions the trace (disjoint and covering), and every mapping set can be produced by some sequence of decisions (working note 4.3).

A rule is a function of a `SweepStep` that returns `True` to absorb. A `SweepStep` has `trace`, `current` (the visit $m(p)$), `entry` ($p$), `start` and `end` of $\xi$, `positions` (of $\xi$) and `inner_visits` ($\{m(e) \mid e \in \xi\}$).

| key | rule | absorbs $\xi$ iff |
|---|---|---|
| S1 | `earlier_wins` | always (coarsest; in an overlap the earlier visit keeps the shared part) |
| S2 | `absorb_never` | never (finest: the uninterrupted runs) |
| S3 | `vehicle_only` | all relations of $\xi$ are on vehicles |
| S4 | `later_wins` | no inner visit overlaps the current one and reaches beyond its end |

b8 = e35, …, e41 at A1, B1, A2, T, B1, A1, B1; Nina's target is (A e35–e37), (T e38), (B e39–e41). The `Segment`s have 0-based positions:

In [ ]:
b8 = traces["b8"]
for rule in (earlier_wins, absorb_never, vehicle_only, later_wins):
    print(f"{rule.__name__:13}", [(s.high_level, b8.events[s.first], b8.events[s.last]) for s in sweep_trace(b8, rule)])

Your own rule is a function like these, e.g. "absorb unless the current visit is at the truck":

In [ ]:
def never_into_vehicles(step: SweepStep) -> bool:
    return step.current.high_level != "T"

[(s.high_level, b8.events[s.first], b8.events[s.last]) for s in sweep_trace(b8, never_into_vehicles)]

### From segments to visits: `segments_frame(segments)` and `mapping_visits(relations, segments)`

`segments_frame` turns (hu, segment) pairs into a frame with positions starting at 1; `mapping_visits` adds the columns of the maximal visits (first, last and exit event, times, duration, counts, interrupted), so a mapping set can be passed to everything that takes visits (visit relations, shifts, waiting times, trace figure).
`map_traces(relations, maximal_visits, strategy)` applies a strategy to every trace and returns the segments frame.

In [ ]:
segments = segments_frame((t.hu, s) for t in traces.values() for s in sweep_trace(t, earlier_wins))
mapping_visits(rel, segments).filter(pl.col("hu") == "b8")

## 4. Global strategies: the best segmentation of the whole trace

### Criteria: `CRITERIA`

Global strategies compare whole segmentations by criteria that are sums over the visits, so a dynamic programme finds the best one.
A segment is a **stay** (it has a stop: a relation at its $hl$ not on a vehicle), a **vehicle visit** (all located relations on vehicles), a **transferring visit** (no stop, but other relations) or a **⊥ visit**.

In [ ]:
pl.DataFrame([(key, c.direction, c.description) for key, c in CRITERIA.items()], schema=["criterion", "direction", "per segment"], orient="row")

### Valid segments, counts and costs: `valid_ends(trace, first)`, `segment_stats(trace, first, last)`, `segment_cost(stats, strategy)`

`valid_ends` gives every $q$ such that $[p, q]$ is a visit: inside $m(p)$, ending at its $hl$, with neighbours not at its $hl$.
`segment_stats` counts a segment (relations, conforming relations, conforming processing relations, kind, duration in hours to the exit event); `segment_cost` turns it into the cost vector of a strategy (lower is better; `None` if the hard rule forbids it).
For b8 at position 0 (e35):

In [ ]:
s6 = nina_strategy()
rows = []
for q in valid_ends(b8, 0):
    stats = segment_stats(b8, 0, q)
    rows.append((f"e35–{b8.events[q]}", stats.kind, stats.n_relations, stats.n_at_location, stats.duration_hours, segment_cost(stats, s6)))
pl.DataFrame(rows, schema=["segment", "kind", "n_relations", "n_at_location", "duration_h", "S6 cost"], orient="row")

### Predefined: S5 `cost_optimal(lam=1.0)` and S6 `nina_strategy(storage_tie_break=True)`

- **S5 cost-optimal**: weighted, $\text{n\_visits} + \lambda \cdot \text{n\_nonconforming}$. Default $\lambda = 1$ (Nina, 2026-11-02), ties broken in favour of longer earlier visits, which gives exactly S1 on this example; $\lambda \in [1.1, 1.4]$ hits all targets except b6, $\lambda \le 1$ all except b8.
- **S6 integrating domain knowledge** (Nina's reasons; called S7 in the working note 6.2, so `"S7"` works as well): hard rule "no sacrifice to vehicles" (a visit without stops must be a vehicle visit), then lexicographically: max conforming processing relations, min stays, min non-conforming relations, min vehicle visits, then the **storage tie-break** (Nina, 2026-11-02; `storage_tie_break=False` switches it off): max relations in stays at a high-level location with a storage unit, then max $\sum d^2$.

`optimal_segmentation(trace, strategy)` returns the segments and the total cost. Ties are broken like S1 (longest first visit first).

In [ ]:
for strategy in (cost_optimal(), cost_optimal(1.25), s6):
    segs, cost = optimal_segmentation(b8, strategy)
    print(f"{strategy.name:28}", [(s.high_level, b8.events[s.first], b8.events[s.last]) for s in segs], cost)

### Your own strategy: `lexicographic(criteria, no_sacrifice_to_vehicles=False)` and `weighted(weights, no_sacrifice_to_vehicles=False)`

| parameter | meaning |
|---|---|
| `criteria` | criterion keys in order of importance; a pair `("duration_sq", "min")` overrides a direction |
| `weights` | criterion key → weight; the weighted sum is signed by the directions, so a positive weight always means "better in its direction" |
| `no_sacrifice_to_vehicles` | the hard rule of S6 on or off |

S6 and S5 are presets of these. Example: fewest visits first, then fewest non-conforming relations, without the hard rule:

In [ ]:
fewest = lexicographic(["n_visits", "n_nonconforming"])
mine = weighted({"n_stays": 1, "n_nonconforming": 0.5, "n_conforming_processing": 2}, no_sacrifice_to_vehicles=True)
for strategy in (fewest, mine):
    print(f"{strategy.name:24}", [(s.high_level, b8.events[s.first], b8.events[s.last]) for s in optimal_segmentation(b8, strategy)[0]])

## 5. All in one call

### `derive_mapping_set(relations, location="loc", agg=None, vehicles=(), kappa=0, processing=None, strategy="S6", storage_units=(), ignore_vehicles=False, trivial_without_processing=False, trivial_excludes_connecting=True)`

Maximal visits, then the strategy on every trace, then the visits table and the metrics. `storage_units` ($W$) is only needed for the storage tie-break of S6. `strategy` is a key (`"S1"` to `"S5"`, `"S6"`), a rule, or a strategy object. Returns a `MappingAnalysis` with `relations`, `maximal_visits`, `mapping`, `metrics` and `strategy` (its name).

### `compare_strategies(relations, strategies=("S1", "S2", "S3", "S4", "S5", "S6"), **kwargs)`

Runs `derive_mapping_set` per strategy; returns the pooled metrics side by side and the analyses by name.

`visit_sequences(mapping)` writes each visit sequence as one text. Nina's targets (working note 4.1): b1 (A e3–e11) (T e13–e14); b2 (A e3–e11) (B e12–e15); b4 (B e15–e21); s2 (A e22–e26); b6 (A e26–e30); b7 (A e26–e33) (B e34) or (A e26) (B e31–e34) (indifferent); b8 (A e35–e37) (T e38) (B e39–e41). S6 hits all of them; its storage tie-break takes the second option of b7 (B holds the storage unit B1), without it S6 takes the first (the longer visit to A).

In [ ]:
OPTIONS = {"location": "loc", "agg": context.agg, "vehicles": context.vehicles, "kappa": KAPPA, "processing": P, "storage_units": context.storage_units}
table, analyses = compare_strategies(relations, **OPTIONS)
sequences = None
for name, analysis in analyses.items():
    part = visit_sequences(analysis.mapping).rename({"sequence": name.split(" ")[0]})
    sequences = part if sequences is None else sequences.join(part, on="hu")
sequences

## 6. Evaluation metrics

### `mapping_metrics(relations, mapping, vehicles=(), processing=None, location="loc", handling_units=None, trivial_without_processing=False, trivial_excludes_connecting=True)`

Takes any mapping set (e.g. `MappingAnalysis.mapping`, or a hand-written table with hu, high_level and first/last event or position) for one or several handling units and returns a `MappingMetrics` with `visits` (per visit: kind, sequence index `j`, loop, connecting, trivial), `per_hu` and `pooled`:

| metric | column | definition |
|---|---|---|
| $N_{vis}$ | n_visits | number of visits |
| $sh_{proc}$ | n_conforming_processing / n_located_processing | processing relations at their visit's $hl$ / located processing relations |
| $N_{loop}$ | n_loops | stays whose $hl$ occurred in an earlier stay, with a stay elsewhere in between (A, T, A is no loop) |
| $N_{triv}$ | n_trivial | visits with one relation (Nina, note l. 143) that are not connecting (`trivial_excludes_connecting=True`, default, Nina 2026-11-02); `trivial_without_processing=True`: only without processing at their $hl$ (note l. 709; still to be decided) |
| $N_{veh}$, $N_{trans}$ | n_vehicle_visits, n_transferring_visits | |
| $sh_{conn}$ | n_connecting / ($N_{veh} + N_{trans}$) | vehicle or transferring visits between two stays at different $hl$ |
| relations per visit, duration | mean_/median_relations, mean_/median_duration | |
| $sh_{conf}$ | n_conforming / n_relations | relations at their visit's $hl$ |

The pooled metrics of all strategies (from `compare_strategies` above; defaults: trivial = one relation, not connecting):

In [ ]:
table

The four combinations of the two trivial options for S6. (T e38) of b8 has one relation, but it is the separator Nina wants, so by default connecting visits are not trivial; the definition at l. 709 of Nina's note (no processing at its $hl$) is still to be decided:

In [ ]:
s6_analysis = analyses[nina_strategy().name]
pl.concat([
    mapping_metrics(s6_analysis.relations, s6_analysis.mapping, context.vehicles, P, trivial_without_processing=w, trivial_excludes_connecting=x)
    .pooled.select(pl.lit(w).alias("trivial_without_processing"), pl.lit(x).alias("trivial_excludes_connecting"), "n_trivial")
    for w in (False, True) for x in (False, True)
])

Per visit and per handling unit (S6):

In [ ]:
s6_analysis.metrics.visits.filter(pl.col("hu").is_in(["b7", "b8"]))

In [ ]:
s6_analysis.metrics.per_hu

For some handling units only (`handling_units`), and for a hand-written mapping of one handling unit (given by events; option 2 of b7):

In [ ]:
print(mapping_metrics(s6_analysis.relations, s6_analysis.mapping, context.vehicles, P, handling_units=["b7", "b8"]).pooled)
hand = pl.DataFrame({"hu": ["b7", "b7"], "high_level": ["A", "B"], "first_event": ["e26", "e31"], "last_event": ["e26", "e34"]})
mapping_metrics(s6_analysis.relations, hand, context.vehicles, P).visits

## 7. Trace figures: maximal visits and mapping sets

The mapping set is drawn as filled bars ("resolved") below the maximal visits (dashed, "maximal"); see `trace_figure.ipynb` for the figure itself.
Two optional displays (both off by default) show the metrics of the mapping set, as computed by `mapping_metrics` and formatted by
- `metric_lines(pooled, keys=LEGEND_METRICS)`: one line per metric for the whole set, written right of the plot below the legend (`trace_figure(..., legend_metrics=..., legend_metrics_title=...)`),
- `trace_metric_lines(per_hu, keys=TRACE_METRICS)`: the metrics of each handling unit, one per line at the right end of its row (`trace_figure(..., trace_metrics=...)`).

Each metric shows its definition (from Nina's note, "Evaluating mapping sets") on mouse-over. Shares are shown as numerator/denominator, durations in hours.

In [ ]:
fig_relations = relations_for_figure(relations, "loc", None, context.agg, context.vehicles, context.storage_units, processing=P)


def figure_visits(analysis):
    maximal_bars = analysis.maximal_visits.select("hu", "high_level", "first_event", "last_event", pl.format("m{}", "visit_id").alias("visit_id"), pl.lit("maximal").alias("group"))
    resolved_bars = analysis.mapping.select("hu", "high_level", "first_event", "last_event", pl.format("v{}", "visit_id").alias("visit_id"), pl.lit("resolved").alias("group"))
    return pl.concat([maximal_bars, resolved_bars])


for key in ("S1", "S4", "S6"):
    analysis = analyses[STRATEGIES[key]().name]
    trace_figure(
        fig_relations,
        figure_visits(analysis),
        title=analysis.strategy,
        legend_metrics=metric_lines(analysis.metrics.pooled),
        legend_metrics_title=f"metrics ({key})",
        trace_metrics=trace_metric_lines(analysis.metrics.per_hu),
    ).show()

## 8. Extra traces

S6 on the extra traces of the working note (6.5 to 6.8). Nina's resolutions: x1 A, T, B, T, A; x2 a tie she called arbitrary, leaning to A, B, B, B, B, B (which the storage tie-break gives); x3 A, A, A, T, B, B; x4 A, A, A, B, B; x5 A, A, A, A, T, B; y1 A, A, A, B, B; y2 A, B, B, B; y3 A, A, A, A, A, B; y4 A, B, B, B, B, B; y5 (A g22–g26) (T g27) (B g28).

In [ ]:
extra_s6 = derive_mapping_set(extra, **OPTIONS, strategy="S6")
visit_sequences(extra_s6.mapping)

In [ ]:
extra_figure = relations_for_figure(extra, "loc", None, context.agg, context.vehicles, context.storage_units, processing=P)
trace_figure(extra_figure, figure_visits(extra_s6), title="Extra traces, S6", trace_metrics=trace_metric_lines(extra_s6.metrics.per_hu)).show()

## 9. Save the mapping set: `save_mapping(mapping, folder, level, kappa, strategy)` and `load_mapping(folder, level, kappa, strategy)`

Writes `visit_mapping_level_{k}_kappa_{κ}_{strategy}.csv` (the columns of a maximal visits file plus `maximal_visit_id`); here the S6 mapping set of the extended running example to `data/mapping_example/derived/`.

In [ ]:
path = save_mapping(s6_analysis.mapping, DERIVED, 3, KAPPA, "S6")
print(path.relative_to(ROOT))
load_mapping(DERIVED, 3, KAPPA, "S6").head()